In [1]:
import cv2
import numpy as np
from cv2 import dnn
print("OpenCV version:", cv2.__version__)


OpenCV version: 4.13.0


In [2]:
#model paths
proto_file = 'Model\colorization_deploy_v2.prototxt'
model_file = 'Model\colorization_release_v2.caffemodel'
points_file = 'Model/pts_in_hull.npy'
print('all files initiated')

all files initiated


In [4]:
net = dnn.readNetFromCaffe(proto_file, model_file)
kernel = np.load(points_file)

In [24]:
#reading and preprocessing the image
img = cv2.imread('paris.jpg')

#normalize the image
img_norm = img.astype('float32') / 255.0

#convert to LAB color space
img_lab = cv2.cvtColor(img_norm, cv2.COLOR_BGR2LAB)


In [25]:
#adding cluster layers
class8 = net.getLayerId('class8_ab')
conv8 = net.getLayerId('conv8_313_rh')
pts = kernel.transpose().reshape(2, 313, 1, 1)
net.getLayer(class8).blobs = [pts.astype('float32')]
net.getLayer(conv8).blobs = [np.full([1, 313], 2.606, dtype='float32')]

In [26]:
#resize the image to 224x224
img_resized = cv2.resize(img_lab, (224, 224))

#split the l channel
l = cv2.split(img_resized)[0]

#subtract 50 from l channel
l = l - 50

#predicting the a and b channels
net.setInput(cv2.dnn.blobFromImage(l))
ab_channel = net.forward()[0, :, :, :].transpose((1, 2, 0))
#resize the predicted a and b channels to original image size
ab_channel_resized = cv2.resize(ab_channel, (img.shape[1], img.shape[0]))

#l channe
l = cv2.split(img_lab)[0]

#join the l channel with predicted a and b channels
colorised = np.concatenate((l[:, :, np.newaxis], ab_channel_resized), axis=2)

#then cnvt lab to bgr
colorised = cv2.cvtColor(colorised, cv2.COLOR_LAB2BGR)
colorised = np.clip(colorised, 0, 1)

#de-normalize the image
colorised = (colorised * 255).astype('uint8')

#resize the colorised image to original size
img = cv2.resize(img, (640,640))
colorised = cv2.resize(colorised, (640,640))

result = np.hstack((img, colorised))
cv2.imshow('Original and Colorised', result)
cv2.waitKey(0)
cv2.destroyAllWindows()

